# Assignment 2



## Section 1: Short Answer Questions (1 pt each)

1. Given a parameter matrix $W \in \mathbb{R}^{d \times d}$ where the input and output dimensions are identical, we apply the standard LoRA mechanism with a low intrinsic rank of $\frac{d}{8}$. What is the ratio between the number of trainable parameters for this specific matrix between direct fine-tuning and fine-tuning with the LoRA mechanism?

**Answer:**

2. We use the paged attention mechanism to store the KV cache. Suppose we have memory space with 10 physical KV-cache blocks, each has 20 slots, and per slot is capable of storing 1 token of KV cache for all layers. If now vLLM is processing 5 queries (each generates at least one token) in parallel during inference, what is the worst-case number of wasted slots in the memory space? (Note: memory that remains unallocated should not be counted as wasted. And we assume each request has at least 20 tokens in prefilling.)

**Answer:**

3. In SGLang, under the batch-processing setting, to increase the cache hit rate, which ordering is preferred for sorting online requests before inference? What is the alternative ordering strategy for offline processing?

**Answer:**



## Section 2: Multi-Head Self-Attention with KV-Cache (11 pts)

In this section, we are going to complete a MHSA implementation optimized with KV-cache using the einsum notation. For the rest of this section, we refer to the batch size as b, sequence length as q or k, number of heads as n, and head hidden dimension as h. The input to the MHSA layer is the tensor x, and the query, key and value weights are denoted as self.w_q, self.w_k & self.w_v respectively

**Specifically, complete the missing code in the `forward` function in `CausalSelfAttention` marked by a "_".**

On a T4, the code should complete in ~20-25 seconds.


**Point Breakdown**
- Problem 1: 3 pts
- Problem 2: 3 pts
- Problem 3: 3 pts
- Problem 4: 2 pts

### Problem

In [ ]:
import math
import inspect
from dataclasses import dataclass

import torch
import torch.nn as nn
from torch.nn import functional as F

class LayerNorm(nn.Module):
    """ LayerNorm but with an optional bias. PyTorch doesn't support simply bias=False """

    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None

    def forward(self, input):
        return F.layer_norm(input, self.weight.shape, self.weight, self.bias, 1e-5)

class CausalSelfAttention(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        # key, query, value projections for all heads, but in a batch
        # For pedagogical purposes only, we initialize them naively.
        self.w_q = nn.Parameter(torch.randn(config.n_embd, config.n_head, config.n_embd // config.n_head, requires_grad=True))
        self.w_k = nn.Parameter(torch.randn(config.n_embd, config.n_head, config.n_embd // config.n_head, requires_grad=True))
        self.w_v = nn.Parameter(torch.randn(config.n_embd, config.n_head, config.n_embd // config.n_head, requires_grad=True))
        # output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        # regularization
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.dropout = config.dropout
        # causal mask to ensure that attention is only applied to the left in the input sequence
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                    .view(1, 1, config.block_size, config.block_size))

    def forward(self, x, kvcache=None):
        B, T, C = x.size() # batch size, sequence length, embedding dimensionality (n_embd)

        ### Problem 1: calculate query, key, values for all heads in batch #
        q = torch.einsum('_, _ -> bqnh', x, self.w_q)
        k = torch.einsum('_, _ -> bknh', x, self.w_k)
        v = torch.einsum('_, _ -> bknh', x, self.w_v)
        ### END ############################################################

        ### Problem 2: Implement KV Cache ##################################
        if kvcache:
            prev_k, prev_v = kvcache
            k = _
            v = _

        new_kvcache = _
        curr_T = _
        ### END ############################################################

        ### Problem 3: Perform QKT Matmul ##################################
        att = torch.einsum('_, _ -> bnqk', q, k)
        ### END ############################################################

        att = att / math.sqrt(k.size(-1))
        if kvcache:
            att = att.masked_fill(torch.ones_like(self.bias[:,:,:T,:curr_T]) == 0, float('-inf'))
        else:
            att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
        att = att.to(torch.float32)
        att = F.softmax(att, dim=-1)
        att = att.to(x.dtype)
        att = self.attn_dropout(att)

        ### Problem 4: Perform AV Matmul ####################################
        y = torch.einsum('_, _ -> bqnh', att, v)
        y = y.contiguous().view(_, _, _)
        ### END ############################################################


        # output projection
        y = self.resid_dropout(self.c_proj(y))
        return y, new_kvcache

### Helpers

In [ ]:
class MLP(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu    = nn.GELU()
        self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x

class Block(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.ln_1 = LayerNorm(config.n_embd, bias=config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = LayerNorm(config.n_embd, bias=config.bias)
        self.mlp = MLP(config)

    def forward(self, x, kvcache=None):
        attn_out, cache_ele = self.attn(self.ln_1(x), kvcache)
        x = x + attn_out
        x = x + self.mlp(self.ln_2(x))
        return x, cache_ele

class GPT(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.vocab_size is not None
        assert config.block_size is not None
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            drop = nn.Dropout(config.dropout),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = LayerNorm(config.n_embd, bias=config.bias),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight # https://paperswithcode.com/method/weight-tying

        # init all weights
        self.apply(self._init_weights)
        # apply special scaled init to the residual projections, per GPT-2 paper
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02/math.sqrt(2 * config.n_layer))

        # report number of parameters
        print("number of parameters: %.2fM" % (self.get_num_params()/1e6,))

    def get_num_params(self, non_embedding=True):
        """
        Return the number of parameters in the model.
        For non-embedding count (default), the position embeddings get subtracted.
        The token embeddings would too, except due to the parameter sharing these
        params are actually used as weights in the final layer, so we include them.
        """
        n_params = sum(p.numel() for p in self.parameters())
        if non_embedding:
            n_params -= self.transformer.wpe.weight.numel()
        return n_params

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None, kvcache=None):
        device = idx.device
        b, t = idx.size()
        assert t <= self.config.block_size, f"Cannot forward sequence of length {t}, block size is only {self.config.block_size}"
        pos = torch.arange(0, t, dtype=torch.long, device=device).unsqueeze(0) # shape (1, t)

        # forward the GPT model itself
        tok_emb = self.transformer.wte(idx) # token embeddings of shape (b, t, n_embd)
        pos_emb = self.transformer.wpe(pos) # position embeddings of shape (1, t, n_embd)
        x = self.transformer.drop(tok_emb + pos_emb)

        if not kvcache:
            kvcache = [None] * self.config.n_layer
        else:
            x = x[:, [-1], :]

        new_kvcache = []
        for block, kvcache_block in zip(self.transformer.h, kvcache):
            x, cache_ele = block(x, kvcache=kvcache_block)
            new_kvcache.append(cache_ele)

        x = self.transformer.ln_f(x)

        if targets is not None:
            # if we are given some desired targets also calculate the loss
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        else:
            # inference-time mini-optimization: only forward the lm_head on the very last position
            logits = self.lm_head(x[:, [-1], :]) # note: using list [-1] to preserve the time dim
            loss = None

        return logits, loss, new_kvcache

    def crop_block_size(self, block_size):
        # model surgery to decrease the block size if necessary
        # e.g. we may load the GPT2 pretrained model checkpoint (block size 1024)
        # but want to use a smaller block size for some smaller, simpler model
        assert block_size <= self.config.block_size
        self.config.block_size = block_size
        self.transformer.wpe.weight = nn.Parameter(self.transformer.wpe.weight[:block_size])
        for block in self.transformer.h:
            if hasattr(block.attn, 'bias'):
                block.attn.bias = block.attn.bias[:,:,:block_size,:block_size]

    def configure_optimizers(self, weight_decay, learning_rate, betas, device_type):
        # start with all of the candidate parameters
        param_dict = {pn: p for pn, p in self.named_parameters()}
        # filter out those that do not require grad
        param_dict = {pn: p for pn, p in param_dict.items() if p.requires_grad}
        # create optim groups. Any parameters that is 2D will be weight decayed, otherwise no.
        # i.e. all weight tensors in matmuls + embeddings decay, all biases and layernorms don't.
        decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
        nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]
        optim_groups = [
            {'params': decay_params, 'weight_decay': weight_decay},
            {'params': nodecay_params, 'weight_decay': 0.0}
        ]
        num_decay_params = sum(p.numel() for p in decay_params)
        num_nodecay_params = sum(p.numel() for p in nodecay_params)
        print(f"num decayed parameter tensors: {len(decay_params)}, with {num_decay_params:,} parameters")
        print(f"num non-decayed parameter tensors: {len(nodecay_params)}, with {num_nodecay_params:,} parameters")
        # Create AdamW optimizer and use the fused version if it is available
        fused_available = 'fused' in inspect.signature(torch.optim.AdamW).parameters
        use_fused = fused_available and device_type == 'cuda'
        extra_args = dict(fused=True) if use_fused else dict()
        optimizer = torch.optim.AdamW(optim_groups, lr=learning_rate, betas=betas, **extra_args)
        print(f"using fused AdamW: {use_fused}")

        return optimizer

    def estimate_mfu(self, fwdbwd_per_iter, dt):
        """ estimate model flops utilization (MFU) in units of A100 bfloat16 peak FLOPS """
        # first estimate the number of flops we do per iteration.
        # see PaLM paper Appendix B as ref: https://arxiv.org/abs/2204.02311
        N = self.get_num_params()
        cfg = self.config
        L, H, Q, T = cfg.n_layer, cfg.n_head, cfg.n_embd//cfg.n_head, cfg.block_size
        flops_per_token = 6*N + 12*L*H*Q*T
        flops_per_fwdbwd = flops_per_token * T
        flops_per_iter = flops_per_fwdbwd * fwdbwd_per_iter
        # express our flops throughput as ratio of A100 bfloat16 peak flops
        flops_achieved = flops_per_iter * (1.0/dt) # per second
        flops_promised = 312e12 # A100 GPU bfloat16 peak flops is 312 TFLOPS
        mfu = flops_achieved / flops_promised
        return mfu

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """
        Take a conditioning sequence of indices idx (LongTensor of shape (b,t)) and complete
        the sequence max_new_tokens times, feeding the predictions back into the model each time.
        """
        kvcache = None
        for _ in range(max_new_tokens):
            # if the sequence context is growing too long we must crop it at block_size
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            # forward the model to get the logits for the index in the sequence
            logits, _, kvcache = self(idx_cond, kvcache=kvcache)
            # pluck the logits at the final step and scale by desired temperature
            logits = logits[:, -1, :] / temperature
            # optionally crop the logits to only the top k options
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            # apply softmax to convert logits to (normalized) probabilities
            probs = F.softmax(logits, dim=-1)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)
            # append sampled index to the running sequence and continue
            idx = torch.cat((idx, idx_next), dim=1)

        return idx

### Model Config

In [ ]:
from contextlib import nullcontext
import torch
import time

# -----------------------------------------------------------------------------
start = "\n" # or "<|endoftext|>" or etc. Can also specify a file, use as: "FILE:prompt.txt"
max_new_tokens = 250 # number of tokens generated in each sample
temperature = 1.0 # 1.0 = no change, < 1.0 = less random, > 1.0 = more random, in predictions
top_k = 200 # retain only the top_k most likely tokens, clamp others to have 0 probability
seed = 1337
device = 'cuda' # examples: 'cpu', 'cuda', 'cuda:0', 'cuda:1', etc.
dtype = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16' # 'float32' or 'bfloat16' or 'float16'
# -----------------------------------------------------------------------------

torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cuda.matmul.allow_tf32 = True # allow tf32 on matmul
torch.backends.cudnn.allow_tf32 = True # allow tf32 on cudnn
device_type = 'cuda' if 'cuda' in device else 'cpu' # for later use in torch.autocast
ptdtype = {'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[dtype]
ctx = nullcontext() if device_type == 'cpu' else torch.amp.autocast(device_type=device_type, dtype=ptdtype)

@dataclass
class GPTConfig:
    block_size: int = 1024
    vocab_size: int = 50304 # GPT-2 vocab_size of 50257, padded up to nearest multiple of 64 for efficiency
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768
    dropout: float = 0.0
    bias: bool = False # True: bias in Linears and LayerNorms, like GPT-2. False: a bit better and faster

# Instatiate new model.
cnfg = GPTConfig()
model = GPT(cnfg)
model.eval()

### Run Code

In [ ]:
## First, we warmup the model. ##
print("Warming up the model...")
for i in range(5):
    warm = torch.randint(0, cnfg.vocab_size, (16, 200), device=device, dtype=torch.long)
    model(warm)

print("Starting generation and timing...")
# Next, run generation and timing.
with torch.inference_mode():
    ## We generate fake data first.
    s = 800
    x = torch.randint(low=0, high=cnfg.vocab_size-1, size=(16, 200), device=device, dtype=torch.long)
    cum_time = 0
    ## Start the timer.
    torch.cuda.synchronize()
    start_time = time.time()
    with ctx:
        y = model.generate(x, s, temperature=temperature, top_k=top_k)
    ## End the timer.
    torch.cuda.synchronize()
    end_time = time.time()
    cum_time += (end_time-start_time)
    print(f'prefill length: {200}, generation length: {s}, time taken: {cum_time:0.4f}')



number of parameters: 123.66M
Warming up the model...
Starting generation and timing...
prefill length: 200, generation length: 800, time taken: 692.9159


## Section 3: Grouped Query Attention with KV-Cache (7 pts)

In this section, we are going to complete code-snippets for a grouped query implementation using the einsum notation. For the rest of this section, we refer to the batch size as b, sequence length as s, number of heads as n, and head hidden dimension as h. Complete the following code-snippets. The input to the GQA layer is the tensor x, and the query, key and value weights are denoted as self.w_q, self.w_k & self.w_v respectively.

**Specifically, complete the missing code in the `forward` function in `CausalSelfAttention` marked by a "_".**

On a T4, the code should complete in <20 seconds.

**Point Breakdown:**
- Problem 1: 1 pts
- Problem 2: 1 pts
- Problem 3: 2 pts
- Problem 4: 2 pts
- Problem 5: 1 pts

### Problem

In [ ]:
import math
import inspect
from dataclasses import dataclass

import torch
import torch.nn as nn
from torch.nn import functional as F

class LayerNorm(nn.Module):
    """ LayerNorm but with an optional bias. PyTorch doesn't support simply bias=False """

    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None

    def forward(self, input):
        return F.layer_norm(input, self.weight.shape, self.weight, self.bias, 1e-5)

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.n_head = config.n_head
        self.head_dim = config.n_embd // config.n_head

        n_kv_head = getattr(config, "n_kv_head", None) or self.n_head
        assert self.n_head % n_kv_head == 0, "n_head must be divisible by n_kv_head"
        self.n_kv_head = n_kv_head
        self.n_rep = self.n_head // self.n_kv_head  # Q heads per KV head

        ### Problem 1: Assign weight matrices WQ, WK, WV ########
        self.w_q = nn.Parameter(torch.randn(_, _, _))
        self.w_k = nn.Parameter(torch.randn(_, _, _))
        self.w_v = nn.Parameter(torch.randn(c_, _, _))
        ### END #################################################

        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)

        self.register_buffer(
            "bias",
            torch.tril(torch.ones(config.block_size, config.block_size)).view(1, 1, config.block_size, config.block_size),
            persistent=False,
        )

    def forward(self, x, kvcache=None):
        B, T, C = x.size()  # C == n_embd
        ### Problem 2: calculate query, key, values for all heads in batch #
        q = torch.einsum('_, _ -> bqnh', x, self.w_q)
        k = torch.einsum('_, _ -> bknh', x, self.w_k)
        v = torch.einsum('_, _ -> bknh', x, self.w_v)
        ### END ############################################################

        ### Problem 3: Implement KV Cache ##################################
        if kvcache:
            prev_k, prev_v = kvcache
            k = _
            v = _

        new_kvcache = _
        curr_T = _
        ### END ############################################################

        ### Problem 4: Perform QKT Matmul ##################################
        qg = q.view(_, _, _, _, _) # [B,T,G,R,D]
        att = torch.einsum('_, _ -> bgtrs', _, _)
        ### END ############################################################
        att = att / math.sqrt(self.head_dim)

        # causal mask (prefill only; decode passes T=1 so no mask needed)
        if kvcache is None:
            causal = self.bias[:, :, :T, :T].unsqueeze(3)  # [1,1,T,1,T]
            att = att.masked_fill(causal == 0, float('-inf'))
        else:
            causal = self.bias[:, :, :T, :curr_T].unsqueeze(3)  # [1,1,T,1,curr_T]
            att = att.masked_fill(causal == 0, float('-inf'))

        att = F.softmax(att.to(torch.float32), dim=-1).to(x.dtype)
        att = self.attn_dropout(att)

        ### Problem 5: Perform AV Matmul ##################################
        yg = torch.einsum('_, _ -> btgrd', att, v) # [B,T,G,R,D]
        # [B,T,G,R,D] -> [B,T,H,D] -> [B,T,C]
        y  = yg.reshape(B, T, _, _).reshape(B, T, _ * _)
        ### END ############################################################

        y = self.resid_dropout(self.c_proj(y))
        return y, new_kvcache

### Model Config

In [ ]:
from contextlib import nullcontext
import torch
import time
from typing import Optional

# -----------------------------------------------------------------------------
start = "\n" # or "<|endoftext|>" or etc. Can also specify a file, use as: "FILE:prompt.txt"
max_new_tokens = 250 # number of tokens generated in each sample
temperature = 1.0 # 1.0 = no change, < 1.0 = less random, > 1.0 = more random, in predictions
top_k = 200 # retain only the top_k most likely tokens, clamp others to have 0 probability
seed = 1337
device = 'cuda' # examples: 'cpu', 'cuda', 'cuda:0', 'cuda:1', etc.
dtype = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16' # 'float32' or 'bfloat16' or 'float16'
# -----------------------------------------------------------------------------

torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cuda.matmul.allow_tf32 = True # allow tf32 on matmul
torch.backends.cudnn.allow_tf32 = True # allow tf32 on cudnn
device_type = 'cuda' if 'cuda' in device else 'cpu' # for later use in torch.autocast
ptdtype = {'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[dtype]
ctx = nullcontext() if device_type == 'cpu' else torch.amp.autocast(device_type=device_type, dtype=ptdtype)

@dataclass
class GPTConfig:
    block_size: int = 1024
    vocab_size: int = 50304 # GPT-2 vocab_size of 50257, padded up to nearest multiple of 64 for efficiency
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768
    dropout: float = 0.0
    bias: bool = False # True: bias in Linears and LayerNorms, like GPT-2. False: a bit better and faster
    n_kv_head: int = 4 # Adjust to use MHA, GQA, or MQA

# Instatiate new model.
cnfg = GPTConfig()
model = GPT(cnfg).to(device=device, dtype=ptdtype)
model.eval()

number of parameters: 110.61M


GPT(
  (transformer): ModuleDict(
    (wte): Embedding(50304, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0-11): 12 x Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_proj): Linear(in_features=768, out_features=768, bias=False)
          (attn_dropout): Dropout(p=0.0, inplace=False)
          (resid_dropout): Dropout(p=0.0, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): MLP(
          (c_fc): Linear(in_features=768, out_features=3072, bias=False)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=3072, out_features=768, bias=False)
          (dropout): Dropout(p=0.0, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=768, out_features=50304, bias=False)
)

### Run Code

In [ ]:
## First, we warmup the model. ##
print("Warming up the model...")
for i in range(5):
    warm = torch.randint(0, cnfg.vocab_size, (16, 200), device=device, dtype=torch.long)
    model(warm)

print("Starting generation and timing...")
# Next, run generation and timing.
with torch.inference_mode():
    ## We generate fake data first.
    s = 800
    x = torch.randint(low=0, high=cnfg.vocab_size-1, size=(16, 200), device=device, dtype=torch.long)
    cum_time = 0
    ## Start the timer.
    torch.cuda.synchronize()
    start_time = time.time()
    with ctx:
        y = model.generate(x, s, temperature=temperature, top_k=top_k)
    ## End the timer.
    torch.cuda.synchronize()
    end_time = time.time()
    cum_time += (end_time-start_time)
    print(f'prefill length: {200}, generation length: {s}, time taken: {cum_time:0.4f}')



Warming up the model...
Starting generation and timing...
prefill length: 200, generation length: 800, time taken: 12.4352


## Section 4: Expert Parallelism Load Balancer (29 pts)


In this section, we are going to implement a in-production system used by Deepseek to load balance experts across GPUs when utilizing expert parallelism.

**Specifically, complete all missing code in the following 4 functions marked by a "_".**

Note: to save compute resources, we recommend you switch to CPU for the remainder of this homework assignment


**Point Breakdown**
- Problem 1.1-1.6: 1 pts each
- Problem 2.1-2.4: 1 pts each
- Problem 3.1-3.14: 1 pts each
- Problem 4.1-4.5: 1 pts each


### Context

**Expert Parallelism** in MoE models sends tokens to different **experts** (small
sub-networks). To run efficiently on multiple GPUs, we **replicate** logical
experts into **physical experts** and **place** those replicas across GPUs so
that:
1. **Load is balanced** (no device gets too many tokens), and
2. **Communication cost is reduced** (prefer intra‑node over inter‑node transfers).

EPLB takes as input per‑expert token loads (one row per MoE layer) and produces:
- `physical_to_logical_map` (which logical expert each physical replica serves),
- `logical_to_physical_map` (for each logical expert, which physical replicas implement it),
- `expert_count` (how many replicas each logical expert receives).

Note: in practice, these token loads are gathered from taking the EMA of observed workloads, but this is outside the scope of the project

### High‑level Components
1. **Balanced Packing**: packs weighted items (expert groups) evenly across bins
   (nodes/GPUs), respecting item counts per bin while keeping weights balanced.
2. **Expert Replication**: greedily assigns extra replicas to the most loaded
   logical expert (normalized by current replica count).
3. **Hierarchical Rebalance**: a 3‑step placement that (a) packs *groups* → nodes,
   (b) replicates experts per node, (c) packs physical replicas → GPUs in that node.
4. **Top‑Level Rebalance**: chooses hierarchical vs global policy based on
   group/node divisibility and produces both directions of the mapping.

### Part 1 — Balanced Packing

**What it does:**  
Packs weighted items into `num_packs` bins so that *each bin contains the same
number of items* and the total weights per bin are as balanced as possible.

The goal is to partition a set of weighted items into a fixed number of “packs” (or bins) such that:
 - Each pack has the same number of items (n / num_packs items per pack).
 - The total weights of each pack are as balanced (close) as possible.

This is like the balanced bin-packing problem — but instead of minimizing leftover space, we want to minimize load imbalance between packs.

**Example:**
```
Input:
weight = torch.tensor([[262., 330., 116., 325.], [231., 280., 516., 129.]])
num_packs = 2

Output:
pack_index = torch.tensor([[1, 0, 0, 1], [1, 1, 0, 0]])
rank_in_pack = torch.tensor([[1, 0, 1, 0], [1, 0, 0, 1]])
```
Explanation: We are packing 4 groups per layer into 2 nodes. So, `groups_per_pack == 2`
For Layer 0, we start with `sorted_weights = [330 (1), 325 (3), 262 (0), 116 (2)]`, and we do step-by-step packing.
- group 1 (330) → pack 0 (330)
- group 3 (325) → pack 1 (325)
- group 0 (262) → pack 1 (325+262)
- group 2 (116) → pack 0 (330+116)

So, `pack_index[0] = [1, 0, 0, 1]` and `rank_in_pack[0] = [1, 0, 1, 0]`

For Layer 1, we start with `sorted_weights = [516 (2), 280 (1), 231 (0), 129 (3)]`, and we do step-by-step packing.
- group 2 (516) → pack 0 (516)
- group 1 (280) → pack 1 (280)
- group 0 (231) → pack 1 (280+231)
- group 3 (129) → pack 0 (516+129)

So, `pack_index[0] = [1, 1, 0, 0]` and `rank_in_pack[0] = [1, 0, 0, 1]`


In [ ]:
from typing import Tuple
import torch

def balanced_packing(weight: torch.Tensor, num_packs: int) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Pack n weighted objects to m packs, such that each bin contains exactly n/m objects and the weights of all packs
    are as balanced as possible.

    Parameters:
        weight: [L, n], the weight of each item
        num_packs: number of packs

    Returns:
        pack_index: [L, n], the pack index of each item
        rank_in_pack: [L, n], the rank of the item in the pack
    """
    num_layers, num_groups = weight.shape

    assert num_groups % num_packs == 0
    groups_per_pack = num_groups // num_packs

    # If each pack is supposed to hold only one item each, return trivial assignment
    if groups_per_pack == 1:
        pack_index = torch.arange(weight.size(-1), dtype=torch.int64, device=weight.device).expand(weight.shape)
        rank_in_pack = torch.zeros_like(weight, dtype=torch.int64)
        return pack_index, rank_in_pack

    ### Problem 1: Sort items by weight in descending order
    indices = weight.float().sort(_, descending=_).indices.cpu()
    ### END ############################################################

    ### Problem 2: Initialize output tensors of same size as weight with placeholder -1 values
    pack_index = torch.full_like(_, fill_value=-1, dtype=torch.int64, device='cpu')
    rank_in_pack = torch.full_like(_, fill_value=-1)
    ### END ############################################################


    for i in range(num_layers):
        # Initialize per-layer bookkeeping structures
        pack_weights = [0] * num_packs
        pack_items = [0] * num_packs
        for group in indices[i]: # Iterate over groups in descending weight order
            ### Problem 3: Find the list all packs that are not yet full
            valid_indices = [i for i in range(num_packs) if _]
            ### END ############################################################

            ### Problem 4: Find the pack with the minimum weight that is not yet full
            pack = _(_, key=pack_weights.__getitem__)
            ### END ############################################################

            assert pack_items[pack] < groups_per_pack # Ensure we never overfill a pack

            ### Problem 5: Assign the group to the selected pack
            pack_index[i, group] = _
            rank_in_pack[i, group] = _
            ### END ############################################################

            ### Problem 6: Assign the group to the selected pack
            pack_weights[pack] += _
            pack_items[pack] += _
            ### END ############################################################

    return pack_index, rank_in_pack

In [ ]:
tokens_per_group = torch.tensor([[262., 330., 116., 325.], [231., 280., 516., 129.]])
num_nodes = 2
num_gpus = 8

group_pack_index, group_rank_in_pack = balanced_packing(tokens_per_group, num_nodes)
assert torch.equal(group_pack_index, torch.tensor([[1, 0, 0, 1], [1, 1, 0, 0]])), "Test case 1 failed! \N{CROSS MARK}"
assert torch.equal(group_rank_in_pack, torch.tensor([[1, 0, 1, 0], [1, 0, 0, 1]])), "Test case 1 failed! \N{CROSS MARK}"

tokens_per_phy = torch.tensor([[ 61.0000,  52.0000,  82.5000,  39.0000,   4.0000,  73.0000,  82.5000, 52.0000],
                                [ 56.0000,  91.5000,  86.0000,  90.0000,  66.0000,  40.0000,  91.5000, 66.0000],
                                [ 93.5000, 157.0000,  86.0000,  86.0000,  16.0000,  27.0000,  93.5000, 86.0000],
                                [ 64.0000,  19.0000,  98.5000,  20.0000,  53.5000, 104.0000,  98.5000, 53.5000]])
pack_index, rank_in_pack = balanced_packing(tokens_per_phy, num_gpus // num_nodes)

assert torch.equal(pack_index, torch.tensor([[3, 3, 0, 0, 1, 2, 1, 2], [0, 0, 3, 2, 3, 1, 1, 2],
                                             [1, 0, 3, 3, 0, 2, 2, 1], [3, 0, 1, 2, 3, 0, 2, 1]])), "Test case 2 failed! \N{CROSS MARK}"
assert torch.equal(rank_in_pack, torch.tensor([[0, 1, 0, 1, 1, 0, 0, 1], [1, 0, 0, 0, 1, 1, 0, 1],
                                               [0, 0, 0, 1, 1, 1, 0, 1], [0, 1, 0, 1, 1, 0, 0, 1]])), "Test case 2 failed! \N{CROSS MARK}"

print("All test cases passed! \N{WHITE HEAVY CHECK MARK}")

### Part 2 - Replicate Experts

**What it does:**  
Greedily assigns extra replicas to logical experts to minimize the maximum
replica load. At each step, it gives the next replica to the expert with the
largest `weight / current_replicas`.

We start with:
 - set of logical experts (e.g., 6 experts per MoE layer).
 - Each expert has a certain load weight (tokens, compute cost, etc.).
 - We want to replicate experts into a larger set of physical experts (e.g., 8 total after replication) so that heavy experts get more replicas and overall load is balanced.

So this function determines:
- Which logical expert each physical replica corresponds to (phy2log).
- The replica index of that physical expert (rank).
- How many total replicas each logical expert ended up with (logcnt).

**Example:**
```
Input:
weight = torch.tensor(
[[ 61, 104, 165,  39,   4,  73],
 [ 56, 183,  86,  90, 132,  40],
 [187, 157, 172,  86,  16,  27],
 [ 64,  19, 197,  20, 107, 104]])

num_phy = 8

Output:
phy2mlog = torch.tensor(
[[0, 1, 2, 3, 4, 5, 2, 1],
 [0, 1, 2, 3, 4, 5, 1, 4],
 [0, 1, 2, 3, 4, 5, 0, 2],
 [0, 1, 2, 3, 4, 5, 2, 4]])

rank = torch.tensor(
[[0, 0, 0, 0, 0, 0, 1, 1],
 [0, 0, 0, 0, 0, 0, 1, 1],
 [0, 0, 0, 0, 0, 0, 1, 1],
 [0, 0, 0, 0, 0, 0, 1, 1]])

logcnt = torch.tensor(
[[1, 2, 2, 1, 1, 1],
 [1, 2, 1, 1, 2, 1],
 [2, 1, 2, 1, 1, 1],
 [1, 1, 2, 1, 2, 1]])
```
Explanation: `num_log = 6, num_redundant = 8-6 = 2`. We need to assign slots 6 and 7 in the physical experts. At iteration 1 (i = 6), get the indices of the logical experts with the highest per-replica load, which is `redundant_indices = [2 (165), 1 (183), 0 (187), 2 (197)]`. We then update the 6th column (zero-indexed) with `[2, 1, 0, 2]`. We update the the 6th column of rank with the rank of the newly added replicas by referencing the appropriate column `logcnt`, which in this iteration is `[1,1,1,1]`. Finally, we update the `logcnt` to reflect we created replicas of these experts, so
```
logcnt = torch.tensor([
    [1, 1, 2, 1, 1, 1],
    [1, 2, 1, 1, 1, 1],
    [2, 1, 1, 1, 1, 1],
    [1, 1, 2, 1, 1, 1]])
```

At iteration 2 (i = 7), get the indices of the logical experts with the highest per-replica load, which is `redundant_indices = [1 (104), 4 (132), 2 (172), 4 (107)]`. We then update the 7th column with `[1, 4, 2, 4]`. We update the the 7th column of rank with the rank of the newly added replicas by referencing the appropriate column `logcnt`, which in this iteration is `[1,1,1,1]`. Finally, we update the `logcnt` to reflect we created replicas of these experts, so
```
logcnt = torch.tensor(
    [[1, 2, 2, 1, 1, 1],
     [1, 2, 1, 1, 2, 1],
     [2, 1, 2, 1, 1, 1],
     [1, 1, 2, 1, 2, 1]])
```

In [ ]:
def replicate_experts(weight: torch.Tensor, num_phy: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:

    """
    Replicate `num_log` experts to `num_phy` replicas, such that the maximum load of all replicas is minimized.

    Parameters:
        weight: [X, num_log]
        num_phy: total number of experts after replication

    Returns:
        phy2log: [X, num_phy], logical expert id of each physical expert
        rank: [X, num_phy], the replica rank
        logcnt: [X, num_log], number of replicas for each logical expert
    """
    n, num_log = weight.shape
    num_redundant = num_phy - num_log # Calculate the number of redundant experts to be created
    assert num_redundant >= 0
    device = weight.device

    phy2log = torch.arange(num_phy, dtype=torch.int64, device=device).repeat(n, 1) # Maps each physical expert to its logical expert ID.
    rank = torch.zeros(n, num_phy, dtype=torch.int64, device=device) # The replica rank of each physical expert.
    logcnt = torch.ones(n, num_log, dtype=torch.int64, device=device) # The number of replicas for each logical expert.
    arangen = torch.arange(n, dtype=torch.int64, device=device) # [0, 1, 2, ..., n-1]
    for i in range(num_log, num_phy):
        ### Problem 1: Get the indices of the logical experts with the highest per-replica load
        per_replica_load = _
        max_per_replica_load = _._(dim=-1)
        redundant_indices = max_per_replica_load.indices
        ### END ############################################################

        ### Problem 2: Assign the selected logical experts to the current physical expert
        phy2log[:, i] = _
        ### END ############################################################

        ### Problem 3: Assign the current replica rank for the selected logical experts to the expert’s current replica count (logcnt)
        rank[:, i] = logcnt[_, _]
        ### END ############################################################

        ### Problem 4: Increment the replica count for the selected logical experts
        logcnt[_, _] += 1
        ### END ############################################################

    return phy2log, rank, logcnt

In [ ]:
tokens_per_mlog = torch.tensor([[ 61., 104., 165.,  39.,   4.,  73.], [ 56., 183.,  86.,  90., 132.,  40.], [187., 157., 172.,  86.,  16.,  27.], [ 64.,  19., 197.,  20., 107., 104.]])
num_physical_experts, num_nodes = 16, 2
phy2mlog, phyrank, mlogcnt = replicate_experts(tokens_per_mlog, num_physical_experts // num_nodes)

assert torch.equal(phy2mlog, torch.tensor([[0, 1, 2, 3, 4, 5, 2, 1], [0, 1, 2, 3, 4, 5, 1, 4],
                                           [0, 1, 2, 3, 4, 5, 0, 2], [0, 1, 2, 3, 4, 5, 2, 4]])), "Test case 1 failed! \N{CROSS MARK}"
assert torch.equal(phyrank, torch.tensor([[0, 0, 0, 0, 0, 0, 1, 1], [0, 0, 0, 0, 0, 0, 1, 1],
                                          [0, 0, 0, 0, 0, 0, 1, 1], [0, 0, 0, 0, 0, 0, 1, 1]])), "Test case 1 failed! \N{CROSS MARK}"
assert torch.equal(mlogcnt, torch.tensor([[1, 2, 2, 1, 1, 1], [1, 2, 1, 1, 2, 1],
                                          [2, 1, 2, 1, 1, 1], [1, 1, 2, 1, 2, 1]])), "Test case 1 failed! \N{CROSS MARK}"

print("All test cases passed! \N{WHITE HEAVY CHECK MARK}")

### Part 3 - Replicate Experts





Hierarchical Rebalance

**What it does:**  
Performs a 3‑stage hierarchical placement:
1) groups → nodes (balanced packing),
2) replicate experts per node,
3) pack physical replicas → GPUs in that node. Returns per‑replica logical IDs and ranks as well as per‑expert replica counts.

Each step uses balanced_packing() or replicate_experts() to achieve optimal load balance at its level.

Walking through an example will take too long, so we gave line-by-line commends and instructions on how to implement the function

In [ ]:
import torch

def rebalance_experts_hierarchical(weight: torch.Tensor, num_physical_experts: int,
                      num_groups: int, num_nodes: int, num_gpus: int):
    """
    Parameters:
        weight: [num_moe_layers, num_logical_experts]
        num_physical_experts: number of physical experts after replication
        num_groups: number of expert groups
        num_nodes: number of server nodes, where the intra-node network (e.g, NVLink) is faster
        num_gpus: number of GPUs, must be a multiple of `num_nodes`

    Returns:
        physical_to_logical_map: [num_moe_layers, num_physical_experts]
        logical_to_physical_map: [num_moe_layers, num_logical_experts, X]
        logical_count: [num_moe_layers, num_logical_experts]
    """
    num_layers, num_logical_experts = weight.shape
    assert num_logical_experts % num_groups == 0
    ### Problem 1: Each group must contain an equal # of experts
    group_size = _ # How many experts per group
    ### END ############################################################

    ### Problem 2: Each node must contain an equal # of groups
    assert num_groups % num_nodes == 0
    groups_per_node = _
    ### END ############################################################

    ### Problem 3: GPUs must divide evenly across nodes nodes, physical experts must divide evenly across GPUs
    assert num_gpus % num_nodes == 0
    assert num_physical_experts % num_gpus == 0
    phy_experts_per_gpu = _
    ### END ############################################################

    # Given a permutation where for index i, i maps to value perm[i], return its inverse mapping
    # Ex: if perm = [2,0,1], then perm[0] -> 2, perm[1] -> 0, perm[2] -> 1
    # inverse would give us result[2] -> 0, result[0] -> 1, result[1] -> 2, so inverse(perm) = [1,2,0]
    def inverse(perm: torch.Tensor) -> torch.Tensor:
        inv = torch.empty_like(perm)
        inv.scatter_(1, perm, torch.arange(perm.size(1), dtype=torch.int64, device=perm.device).expand(perm.shape))
        return inv

    # Step 1: pack groups to nodes
    ### Problem 4: Group weights into a Tensor of shape [num_layers, num_groups, group_size] and sum over group_size dimension
    grouped_weights = _.unflatten(-1, (_, _))
    tokens_per_group = _
    ### END ############################################################

    ### Problem 5: Use the balance packing algorithm to assign groups to nodes
    group_pack_index, group_rank_in_pack = balanced_packing(_, _)
    ### END ############################################################

    # Compute a mapping from original logical experts to mixed-node-local logical experts, and its inverse
    # Ex: if node 0 has groups [1, 3], and each group has 3 experts, then its experts occupy slots [0–2] and [3–5] in node 0’s local expert index space.
    log2mlog = (((group_pack_index * groups_per_node + group_rank_in_pack) * group_size).unsqueeze(-1) +
                torch.arange(group_size, dtype=torch.int64, device=group_pack_index.device)).flatten(-2)
    mlog2log = inverse(log2mlog)

    ### Problem 6: Reorder experts per node using mlog2log, so each row now represents all logical experts belonging to one node.
    # The final tensor shape should be [num_layers * num_nodes, num_logical_experts // num_nodes]
    tokens_per_mlog = weight.gather(-1, _).view(-1, _)
    ### END ############################################################

    ### Problem 7: Replicate experts within each node to achieve balanced load
    phy2mlog, phyrank, mlogcnt = replicate_experts(_, _)
    ### END ############################################################

    ### Problem 8: Compute per-replica load: divide each expert’s weight (tokens_per_mlog) by its number of replicas
    per_replica_load = _
    ### END ############################################################

    ### Problem 9: gather using phy2mlog mapping to match physical expert order
    tokens_per_phy = (_).gather(-1, _)
    ### END ############################################################

    ### Problem 10: Assign (pack) physical experts to GPUs within each node
    pack_index, rank_in_pack = balanced_packing(_, _)  # [num_layers * num_nodes, num_physical_experts // num_nodes]
    ### END ############################################################

    ### Problem 11: Compute physical->gpu-local mapping and its inverse
    phy2pphy = _ * _ + _
    pphy2phy = inverse(_)
    ### END ############################################################

    ### Problem 12: Remap all mapping back to global expert space
    pphy2mlog = phy2mlog.gather(-1, _) # [num_layers * num_nodes, num_log_per_nodes]
    ### END ############################################################

    # Add the node offset back to the node-local logical expert indices to merge all nodes
    pphy2mlog = (pphy2mlog.view(num_layers, num_nodes, -1) +
                 torch.arange(0, num_logical_experts, num_logical_experts // num_nodes,
                              device=group_pack_index.device).view(1, -1, 1)).flatten(-2)

    ### Problem 13: Convert node-local expert indices back to global logical expert indices
    pphy2log = mlog2log.gather(-1, _)
    ### END ############################################################

    ### Problem 14: Gather replica ranks and logical expert counts
    pphyrank = phyrank.gather(-1, _).view(_, -1)
    logcnt = mlogcnt.view(_, -1).gather(-1, _)
    ### END ############################################################

    return pphy2log, pphyrank, logcnt

### Part 4 — Top‑Level Rebalance Entry Point

**What it does:**  
Chooses hierarchical or global policy (fallback uses hierarchical with `1 group` and `1 node`),
produces `physical_to_logical_map`, `logical_to_physical_map`, and `expert_count` tensors.

The goal of rebalance_experts() is to return three things that completely describe how experts are placed and replicated across devices:
 - phy2log [layers, num_replicas]: physical → logical mapping (which logical expert each physical one corresponds to)
 - log2phy [layers, num_logical_experts, X]: logical → physical mapping (which physical experts belong to each logical one)
 - logcnt [layers, num_logical_experts]: number of replicas per logical expert

In [ ]:
from typing import Tuple
import torch

def rebalance_experts(weight: torch.Tensor, num_replicas: int, num_groups: int,
                      num_nodes: int, num_gpus: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Entry point for expert-parallelism load balancer.

    Parameters:
        weight: [layers, num_logical_experts], the load statistics for all logical experts
        num_replicas: number of physical experts, must be a multiple of `num_gpus`
        num_groups: number of expert groups
        num_nodes: number of server nodes, where the intra-node network (e.g, NVLink) is faster
        num_gpus: number of GPUs, must be a multiple of `num_nodes`

    Returns:
        physical_to_logical_map: [layers, num_replicas], the expert index of each replica
        logical_to_physical_map: [layers, num_logical_experts, X], the replica indices for each expert
        expert_count: [layers, num_logical_experts], number of physical replicas for each logical expert
    """
    num_layers, num_logical_experts = weight.shape
    weight = weight.float().cpu()

    # If the number of groups divides evenly across nodes, we can exploit hierarchical balance
    if num_groups % num_nodes == 0:
        ### Problem 1: Run the full 3-stage load balancer (group packing → intra-node replication → GPU packing).
        phy2log, phyrank, logcnt = rebalance_experts_hierarchical(_, _, _, _, _)
        ### END ############################################################
    else: # Fall back to global balancing
        ### Problem 2: Treat all experts as belonging to a single node and group
        phy2log, phyrank, logcnt = rebalance_experts_hierarchical(_, _, _, _, _)
        ### END ############################################################

    maxlogcnt = logcnt.max().item()

    ### Problem 3: Initialize log2phy tensor: [num_layers, num_logical_experts, maxlogcnt].
    log2phy = torch.full((_, _, _), -1, dtype=torch.int64, device=logcnt.device)
    ### END ############################################################

    # Compute target scatter index
    target_scatter_index = phy2log * maxlogcnt + phyrank

    ### Problem 4: Get the indices of the physical experts and repeat for each layer
    physical_indices = torch.arange(_, dtype=torch.int64, device=log2phy.device).expand(_, -1)
    ### END ############################################################

    ### Problem 5: for each layer, we write each physical expert’s index into the correct (logical, rank) slot.
    log2phy.view(_, -1).scatter_(-1, _, _)
    ### END ############################################################
    return phy2log, log2phy, logcnt

__all__ = ['rebalance_experts']

In [ ]:
weight = torch.tensor([[ 90, 132,  40,  61, 104, 165,  39,   4,  73,  56, 183,  86],
                       [ 20, 107, 104,  64,  19, 197, 187, 157, 172,  86,  16,  27]])

num_replicas = 16
num_groups = 4
num_nodes = 2
num_gpus = 8

phy2log, log2phy, logcnt = rebalance_experts(weight, num_replicas, num_groups, num_nodes, num_gpus)
assert torch.equal(phy2log, torch.tensor([[ 5,  6,  5,  7,  8,  4,  3,  4, 10,  9, 10,  2,  0,  1, 11,  1],
                                          [ 7, 10,  6,  8,  6, 11,  8,  9,  2,  4,  5,  1,  5,  0,  3,  1]])), "Test case 1 failed! \N{CROSS MARK}"
print("All test case passed! \N{WHITE HEAVY CHECK MARK}")